In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
sys.path.insert(1, '/home/jw3514/Work/UNIMED/src')
from CellType_PSY import *
#from UNIMED import *

try:
    os.chdir(f"{ProjDIR}/notebooks/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

# Try Spec Matrix

In [ ]:
HumanCT_Z2_HCT = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/Test.BiasMat/HumanCT.Spec.clip.csv", index_col=0)
HumanCT_Z2_HCT.columns = [int(x) for x in HumanCT_Z2_HCT.columns.values]

In [ ]:
HumanCT_Z2_HCT.head(2)

In [ ]:
GeneWeightDIR = "../dat/GeneWeights/"
HIQ_GW = Fil2Dict("{}/HIQ.top61.nopLI.LGD_Dmis_SameWeight.gw".format(GeneWeightDIR))
SCZ_GW = Fil2Dict("{}/SCZ.top61.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw".format(GeneWeightDIR))

In [ ]:
HIQ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, HIQ_GW)
HIQ_Bias = AnnotateCTDat(HIQ_Bias, Anno)
SCZ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, SCZ_GW)
SCZ_Bias = AnnotateCTDat(SCZ_Bias, Anno)

Combined_Bias = HIQ_Bias.add_suffix('_HIQ').join(SCZ_Bias.add_suffix('_SCZ'))
Combined_Bias["DiffBias"] = Combined_Bias["EFFECT_HIQ"] - Combined_Bias["EFFECT_SCZ"]

In [ ]:
def swap_dict_keys(dict1, dict2):
    """
    Randomly split all keys from both dictionaries into two new dictionaries
    
    Args:
        dict1: First dictionary
        dict2: Second dictionary
        
    Returns:
        new_dict1, new_dict2: New dictionaries with randomly split keys from the pooled keys
    """
    # Pool all keys and values together
    all_keys = list(dict1.keys()) + list(dict2.keys())
    all_values = list(dict1.values()) + list(dict2.values())
    
    # Get sizes of original dicts
    size1 = len(dict1)
    size2 = len(dict2)
    
    # Randomly shuffle indices
    indices = np.arange(len(all_keys))
    np.random.shuffle(indices)
    
    # Split into two new dictionaries
    new_dict1 = {all_keys[i]: all_values[i] for i in indices[:size1]}
    new_dict2 = {all_keys[i]: all_values[i] for i in indices[size1:]}
    
    return new_dict1, new_dict2


In [ ]:

# Test the function
# Store all permutations
all_perm_biases = []

# Run 1000 permutations
for i in range(1000):
    new_hiq, new_scz = swap_dict_keys(HIQ_GW, SCZ_GW)
    
    Perm_HIQ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, new_hiq)
    Perm_HIQ_Bias = AnnotateCTDat(Perm_HIQ_Bias, Anno)
    Perm_SCZ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, new_scz)
    Perm_SCZ_Bias = AnnotateCTDat(Perm_SCZ_Bias, Anno)
    
    Combined_Perm_Bias = Perm_HIQ_Bias.add_suffix('_HIQ').join(Perm_SCZ_Bias.add_suffix('_SCZ'))
    Combined_Perm_Bias["DiffBias"] = Combined_Perm_Bias["EFFECT_HIQ"] - Combined_Perm_Bias["EFFECT_SCZ"]
    
    all_perm_biases.append(Combined_Perm_Bias)



In [ ]:
Combined_Bias.head(2)

In [ ]:
def plot_supercluster_effect(supercluster, combined_bias, all_permutations):
    """
    Plot effect size distribution and calculate p-value for a given supercluster
    
    Args:
        supercluster (str): Name of supercluster to analyze
        combined_bias (pd.DataFrame): DataFrame with observed effects
        all_permutations (list): List of DataFrames with permuted effects
        
    Returns:
        tuple: (matplotlib figure, p-value)
    """
    # Calculate observed effect
    obs = combined_bias[combined_bias["Supercluster_HIQ"]==supercluster]["DiffBias"].mean()

    # Calculate null distribution
    null = []
    for df in all_permutations:
        eff = df[df["Supercluster_HIQ"]==supercluster]["DiffBias"].mean()
        null.append(eff)
    null = np.array(null)

    # Create plot
    fig = plt.figure(figsize=(10, 6))
    plt.hist(null, bins=30, alpha=0.5, label='Null')
    plt.axvline(obs, color='red', linestyle='--', label='Observed')
    plt.xlabel('Effect Size')
    plt.ylabel('Count')
    plt.title(f'{supercluster} Effect Sizes')
    plt.legend()
    plt.tight_layout()

    # Calculate p-value
    p_value = np.mean(null >= obs) if obs >= 0 else np.mean(null <= obs)
    
    return fig, p_value

# add another function test indiviudal cluster belongs to a supercluster
def plot_cluster_effect(supercluster, combined_bias, all_permutations):
    """
    Test if a given supercluster is enriched for a specific effect size 

    Args:
        supercluster (str): Name of supercluster to analyze
        combined_bias (pd.DataFrame): DataFrame with observed effects
        all_permutations (list): List of DataFrames with permuted effects
        
    Returns:
        tuple: (matplotlib figure, p-value) 
    """
    test_DF = combined_bias[combined_bias["Supercluster_HIQ"]==supercluster]
    for cluster in test_DF.index.values:
        # Calculate observed effect
        obs = test_DF.loc[cluster, "DiffBias"]

        # Calculate null distribution
        null = []
        for df in all_permutations:
            eff = df.loc[cluster, "DiffBias"]
            null.append(eff)
        null = np.array(null)

        # Calculate p-value
        p_value = np.mean(null >= obs) if obs >= 0 else np.mean(null <= obs)
        print(f"P-value for {cluster}: {p_value}")


In [ ]:
Supercluster = "CGE interneuron"
plot_supercluster_effect(Supercluster, Combined_Bias, all_perm_biases)
plot_cluster_effect(Supercluster, Combined_Bias, all_perm_biases)

In [ ]:
Supercluster = "Medium spiny neuron"
plot_supercluster_effect(Supercluster, Combined_Bias, all_perm_biases)
plot_cluster_effect(Supercluster, Combined_Bias, all_perm_biases)

In [ ]:
RandG_SCZ_top61_DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/RandGeneBias_Spec_clip/RandGenes_SCZ.top61_May28/all/"
RandG_SCZ_top61_DFs = LoadNullDF(RandG_SCZ_top61_DIR)
RandG_ASD_HIQ_top61_DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/RandGeneBias_Spec_clip/RandGenes_ASD_HIQ.top61_May28/all/"
RandG_ASD_HIQ_top61_DFs = LoadNullDF(RandG_ASD_HIQ_top61_DIR)

In [ ]:
BiasDiff_Null = []
Supercluster = "CGE interneuron"
Supercluster_idx = Anno[Anno["Supercluster"]==Supercluster].index
for i in range(1000):
    df1 = RandG_SCZ_top61_DFs[i]
    df2 = RandG_ASD_HIQ_top61_DFs[i]
    df1_supercluster = df1.loc[Supercluster_idx, 'EFFECT'].mean()
    df2_supercluster = df2.loc[Supercluster_idx, 'EFFECT'].mean()
    BiasDiff_Null.append(df1_supercluster - df2_supercluster)
BiasDiff_Null = np.array(BiasDiff_Null)
BiasDiff_Null.mean()


In [ ]:
obs

In [ ]:
obs = Combined_Bias.loc[Supercluster_idx, "DiffBias"].mean()
plt.hist(BiasDiff_Null, bins=30, alpha=0.5, label='Null')
plt.axvline(obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Bias Difference')
plt.ylabel('Count')
plt.title(f'{Supercluster} Bias Difference')
plt.legend()
plt.tight_layout()

# [End of Try Spec Matrix]

In [ ]:
HumanCTExpL = pd.read_csv("../dat/HumanCTExpressionMats/Human.Cluster.Log2Mean.Exp.csv", index_col=0)
HumanCTExpL.columns = [int(x) for x in HumanCTExpL.columns.values]

In [ ]:
HumanCT_Z2 = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/Human.Cluster.Log2Mean.Z1clip5.Z2.clip2.Jan21.csv", index_col=0)
HumanCT_Z2.columns = [int(x) for x in HumanCT_Z2.columns.values]

In [ ]:
def test_supercluster_enrichment(supercluster_to_test, ObsDF, RandDFs):
    supercluster_idx = ObsDF[ObsDF["Supercluster"]==supercluster_to_test].index
    Obs = ObsDF.loc[supercluster_idx, "EFFECT"].mean()
    print(f"Observed effect for {supercluster_to_test}: {Obs}")

    Null = []
    for df in RandDFs:
        eff = df.loc[supercluster_idx, "EFFECT"].mean()
        Null.append(eff)
    Null = np.array(Null)

    Z, P = GetPermutationP(Null, Obs)
    print(f"P-value: {P}")
    return Obs, Null, Z, P



# Weight Redistribution Test (Jan 17 2025)

####  ASD Type1 (Randomly redistribute the mutations, and assign new weights to same genes)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_HIQ_Type1/"
Perm_Z2_BiasDFs_ASD_HIQ_Type1 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_HIQ_Type1.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_LIQ_Type1/"
Perm_Z2_BiasDFs_ASD_LIQ_Type1 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_LIQ_Type1.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_ALL_Type1/"
Perm_Z2_BiasDFs_ASD_ALL_Type1 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_ALL_Type1.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_ExomeWide_Type1/"
Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type1 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type1.append(Perm_Z2_Bias)

In [ ]:
BiasDIR = "/home/jw3514/Work/CellType_Psy/dat/Bias_Jan13_2025/"
ASD_HIQ_Bias = pd.read_csv("{}/HCT.ASD61.HIQ.Z2.HCT.csv".format(BiasDIR), index_col=0)
ASD_LIQ_Bias = pd.read_csv("{}/HCT.ASD61.LIQ.Z2.HCT.csv".format(BiasDIR), index_col=0)
ASD_ALL_Bias = pd.read_csv("{}/HCT.ASD61.Z2.HCT.csv".format(BiasDIR), index_col=0)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_HIQ_Bias, Perm_Z2_BiasDFs_ASD_HIQ_Type1)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_LIQ_Bias, Perm_Z2_BiasDFs_ASD_LIQ_Type1)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_Type1)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type1)

In [ ]:
#Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE3)
CT = "Medium spiny neuron"
Obs, Null, Z, P = test_supercluster_enrichment(CT, ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type1)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('Medium spiny neuron [Type 1 Perm]\nP = {:.3f}'.format(P))
plt.legend()
plt.tight_layout()

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_HIQ_Bias, Perm_Z2_BiasDFs_ASD_HIQ_Type1)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_LIQ_Bias, Perm_Z2_BiasDFs_ASD_LIQ_Type1)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_Type1)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type1)

####  ASD HIQ Type2 (Randomly redistribute the mutations according to background mutation rate, and assign new weights to same genes)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_HIQ_Type2/"
Perm_Z2_BiasDFs_ASD_HIQ_Type2 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_HIQ_Type2.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_LIQ_Type2/"
Perm_Z2_BiasDFs_ASD_LIQ_Type2 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_LIQ_Type2.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_ALL_Type2/"
Perm_Z2_BiasDFs_ASD_ALL_Type2 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_ALL_Type2.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_ExomeWide_Type2/"
Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type2 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type2.append(Perm_Z2_Bias)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_HIQ_Bias, Perm_Z2_BiasDFs_ASD_HIQ_Type2)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_LIQ_Bias, Perm_Z2_BiasDFs_ASD_LIQ_Type2)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_Type2)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type2)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_HIQ_Bias, Perm_Z2_BiasDFs_ASD_HIQ_Type2)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_LIQ_Bias, Perm_Z2_BiasDFs_ASD_LIQ_Type2)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_Type2)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type2)

In [ ]:
# Create empty lists to store results
superclusters_list = []
pvalues_list = []
bias_list = []
null_mean_list = []
# Test each supercluster and save results
for supercluster in ALL_CTs:
    Obs, Null, Z, P = test_supercluster_enrichment(supercluster, ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type2)
    superclusters_list.append(supercluster)
    pvalues_list.append(P)
    bias_list.append(Obs)
    null_mean_list.append(Null.mean())
# Create dataframe with results
results_df = pd.DataFrame({
    'Supercluster': superclusters_list,
    'bias': bias_list,
    'null_mean': null_mean_list,
    'P_value': pvalues_list
})

# Calculate FDR-corrected p-values
results_df['FDR'] = multipletests(results_df['P_value'], method='fdr_bh')[1]

# Sort by FDR-corrected p-value
results_df = results_df.sort_values('FDR')

In [ ]:
results_df

In [ ]:
0.05 / 32 / 2

####  ASD HIQ Type3 (Randomly redistribute the mutations according to background mutation rate, and assign new weights to random genes)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_HIQ_Type3/"
Perm_Z2_BiasDFs_ASD_HIQ_Type3 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_HIQ_Type3.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_LIQ_Type3/"
Perm_Z2_BiasDFs_ASD_LIQ_Type3 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_LIQ_Type3.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_ALL_Type3/"
Perm_Z2_BiasDFs_ASD_ALL_Type3 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_ALL_Type3.append(Perm_Z2_Bias)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_ExomeWide_Type3/"
Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type3 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type3.append(Perm_Z2_Bias)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_HIQ_Bias, Perm_Z2_BiasDFs_ASD_HIQ_Type3)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_LIQ_Bias, Perm_Z2_BiasDFs_ASD_LIQ_Type3)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_Type3)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type3)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_HIQ_Bias, Perm_Z2_BiasDFs_ASD_HIQ_Type3)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_LIQ_Bias, Perm_Z2_BiasDFs_ASD_LIQ_Type3)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_Type3)
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", ASD_ALL_Bias, Perm_Z2_BiasDFs_ASD_ALL_ExomeWide_Type3)

In [ ]:
ASD_LIQ_Bias.head(2)

In [ ]:

def test_customize_set_enrichment(cell_indice, ObsDF, RandDFs):
    Obs = ObsDF.loc[cell_indice, "EFFECT"].mean()
    print(f"Observed effect for cell types: {Obs}")
    Null = []
    for df in RandDFs:
        eff = df.loc[cell_indice, "EFFECT"].mean()
        Null.append(eff)
    Null = np.array(Null)

    Z, P = GetPermutationP(Null, Obs)
    print(f"P-value: {P}")
    return Obs, Null, Z, P

In [ ]:
test_cell_idx = ASD_LIQ_Bias[ASD_LIQ_Bias["Supercluster"] == "CGE interneuron"].head(5).index.values

In [ ]:
Obs, Null, Z, P =  test_customize_set_enrichment(test_cell_idx, ASD_LIQ_Bias, Perm_Z2_BiasDFs_ASD_LIQ_Type3)

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('Medium spiny neuron [Type 3 Perm]\nP = {:.3f}'.format(P))
plt.legend()
plt.tight_layout()


In [ ]:
CellType = "Medium spiny neuron"
cellType_idx = Anno[Anno["Supercluster"] == CellType].index
#print(cellType_idx)
ASD_supercluster_biases = ASD_ALL_Bias.loc[cellType_idx, :]["EFFECT"]

In [ ]:
Rand_gene_idx_gt_asd = []
test_dat = []
counter = 0
for i in range(len(Perm_Z2_BiasDFs_ASD_ALL_Type3)):
    df = Perm_Z2_BiasDFs_ASD_ALL_Type3[i]
    df_cellType = df.loc[cellType_idx, :]
    if df_cellType["EFFECT"].mean() > ASD_supercluster_biases.mean():
        Rand_gene_idx_gt_asd.append(i)
        test_dat.append(df_cellType["EFFECT"])
        print(i)
        counter += 1
    if counter > 5:
        break
print(counter)
test_dat = np.array(test_dat)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.boxplot([ASD_supercluster_biases] + [test_dat[i] for i in range(len(test_dat))], 
            labels=['ASD'] + [f'Random {i+1}' for i in range(len(test_dat))])
plt.ylabel('Effect Size')
plt.title('ASD vs Random Gene Sets Effect Sizes for Medium Spiny Neurons')
plt.xticks(rotation=45)
plt.show()

print("\nMean values:")
print(f"ASD: {ASD_supercluster_biases.mean():.3f}")
for i in range(len(test_dat)):
    print(f"Random {i+1}: {test_dat[i].mean():.3f}")

In [ ]:
ASD_GW = Fil2Dict("../../ASD_Circuits/dat/Unionize_bias/Spark_Meta_EWS.GeneWeight.csv")
asd_Genes = list(ASD_GW.keys())

In [ ]:
tmp_idx = 59
tmp_GW = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/ASD_ALL_Type3/{}.perm.GW.txt".format(tmp_idx), index_col=0, header=None)
tmp_Genes = tmp_GW.index.values



In [ ]:
CellType = "Medium spiny neuron"
cellType_idx = Anno[Anno["Supercluster"] == CellType].index
print(cellType_idx)

super_cluster_biases = HumanCT_Z2.loc[:, cellType_idx].mean(axis=1)
super_cluster_ExpL = HumanCTExpL.loc[:, cellType_idx].mean(axis=1)

asd_genes_cluster_biases = HumanCT_Z2.loc[asd_Genes, cellType_idx].mean(axis=1)
asd_genes_cluster_ExpL = HumanCTExpL.loc[asd_Genes, cellType_idx].mean(axis=1)

rand_genes_cluster_biases = HumanCT_Z2.loc[tmp_Genes, cellType_idx].mean(axis=1)
rand_genes_cluster_ExpL = HumanCTExpL.loc[tmp_Genes, cellType_idx].mean(axis=1)


In [ ]:

plt.figure(figsize=(12,8))
plt.scatter(super_cluster_biases, np.log2(super_cluster_ExpL+1), s=0.1, color="grey")
plt.scatter(asd_genes_cluster_biases, np.log2(asd_genes_cluster_ExpL+1), s=5, color="red")
plt.scatter(rand_genes_cluster_biases, np.log2(rand_genes_cluster_ExpL+1), s=5, color="blue")
plt.axvline(x=0, color='black', linestyle='--')

n_positive = sum(asd_genes_cluster_biases > 0)
n_negative = sum(asd_genes_cluster_biases < 0)
plt.text(0.02, 0.98, f'ASD genes Z2>0: {n_positive}\nASD genes Z2<0: {n_negative}', 
         transform=plt.gca().transAxes, verticalalignment='top')
n_positive = sum(rand_genes_cluster_biases > 0)
n_negative = sum(rand_genes_cluster_biases < 0)
plt.text(0.02, 0.80, f'Random genes Z2>0: {n_positive}\nRandom genes Z2<0: {n_negative}', 
         transform=plt.gca().transAxes, verticalalignment='top')


### SCZ 

In [ ]:
BiasDIR = "/home/jw3514/Work/CellType_Psy/dat/Bias_Jan13_2025/"
SCZ_Bias = pd.read_csv("{}/HCT.SCZ61.Z2.HCT.csv".format(BiasDIR), index_col=0)

### Type 1: Randomly redistribute the mutations, and assign new weights to same genes

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/SCZ_Type1/"
SCZ_Perm_Z2_BiasDFs_TYPE1 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    SCZ_Perm_Z2_BiasDFs_TYPE1.append(Perm_Z2_Bias)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE1)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE1)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE1)

In [ ]:
CT = "CGE interneuron"  
Obs, Null, Z, P = test_supercluster_enrichment(CT, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE1)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('Medium spiny neuron [Type 3 Perm]\nP = {:.3f}'.format(P))
plt.legend()
plt.tight_layout()

In [ ]:
CT = "MGE interneuron"  
Obs, Null, Z, P = test_supercluster_enrichment(CT, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE1)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('Medium spiny neuron [Type 3 Perm]\nP = {:.3f}'.format(P))
plt.legend()
plt.tight_layout()

### Type 2: Randomly redistribute the mutations according to background mutation rate, and assign new weights to same genes

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/SCZ_Type2/"
SCZ_Perm_Z2_BiasDFs_TYPE2 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    SCZ_Perm_Z2_BiasDFs_TYPE2.append(Perm_Z2_Bias)


In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE2)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE2)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE2)

In [ ]:
# Create empty lists to store results
superclusters_list = []
pvalues_list = []

# Test each supercluster and save results
for supercluster in ALL_CTs:
    Obs, Null, Z, P = test_supercluster_enrichment(supercluster, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE2)
    superclusters_list.append(supercluster)
    pvalues_list.append(P)

# Create dataframe with results
results_df = pd.DataFrame({
    'Supercluster': superclusters_list,
    'P_value': pvalues_list
})

# Calculate FDR-corrected p-values
results_df['FDR'] = multipletests(results_df['P_value'], method='fdr_bh')[1]

# Sort by FDR-corrected p-value
results_df = results_df.sort_values('FDR')

In [ ]:
results_df

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('Medium spiny neuron [Type 3 Perm]\nP = {:.3f}'.format(P))
plt.legend()
plt.tight_layout()

### Type 3: Randomly redistribute the mutations according to background mutation rate, and assign new weights to same genes

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/SCZ_Type3/"
SCZ_Perm_Z2_BiasDFs_TYPE3 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    SCZ_Perm_Z2_BiasDFs_TYPE3.append(Perm_Z2_Bias)


In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE3)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE3)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE3)

In [ ]:
#Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE3)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE3)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('Medium spiny neuron [Type 3 Perm]\nP = {:.3f}'.format(P))
plt.legend()
plt.tight_layout()

### type 4 

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/SCZ_Type4/"
SCZ_Perm_Z2_BiasDFs_TYPE4 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    SCZ_Perm_Z2_BiasDFs_TYPE4.append(Perm_Z2_Bias)


In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)

In [ ]:
#Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
CT = "MGE interneuron"  
Obs, Null, Z, P = test_supercluster_enrichment(CT, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
#CT = "MGE interneuron"
#Obs, Null, Z, P = test_supercluster_enrichment(CT, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('{} [Type 4 Perm]\nP = {:.3f}'.format(CT, P))
plt.legend()
plt.tight_layout()

### NDD 

In [ ]:
BiasDIR = "/home/jw3514/Work/CellType_Psy/dat/Bias_Jan13_2025/"
NDD_Bias = pd.read_csv("{}/HCT.DDDHC.Z2.HCT.top61.csv".format(BiasDIR), index_col=0)

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/NDD_Type1/"
NDD_Perm_Z2_BiasDFs_TYPE1 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    NDD_Perm_Z2_BiasDFs_TYPE1.append(Perm_Z2_Bias)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE1)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE1)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE1)

In [ ]:
#Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
CT = "CGE interneuron"  
#CT = "MGE interneuron"  
Obs, Null, Z, P = test_supercluster_enrichment(CT, NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE1)
#CT = "MGE interneuron"
#Obs, Null, Z, P = test_supercluster_enrichment(CT, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('{} [Type 3 Perm]\nP = {:.3f}'.format(CT, P))
plt.legend()
plt.tight_layout()

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/NDD_Type2/"
NDD_Perm_Z2_BiasDFs_TYPE2 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    NDD_Perm_Z2_BiasDFs_TYPE2.append(Perm_Z2_Bias)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE2)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE2)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE2)

In [ ]:
#Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
#CT = "CGE interneuron"  
CT = "MGE interneuron"  
Obs, Null, Z, P = test_supercluster_enrichment(CT, NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE2)
#CT = "MGE interneuron"
#Obs, Null, Z, P = test_supercluster_enrichment(CT, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('{} [Type 3 Perm]\nP = {:.3f}'.format(CT, P))
plt.legend()
plt.tight_layout()

In [ ]:
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/WeightRedistribute_Jan17/NDD_Type3/"
NDD_Perm_Z2_BiasDFs_TYPE3 = []
for i in np.arange(1000):
    Perm_Z2_Bias = pd.read_csv("{}/{}.perm.Z2.csv.gz".format(DIR, i), index_col=0)
    NDD_Perm_Z2_BiasDFs_TYPE3.append(Perm_Z2_Bias)

In [ ]:
Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE3)
Obs, Null, Z, P = test_supercluster_enrichment("MGE interneuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE3)
Obs, Null, Z, P = test_supercluster_enrichment("Medium spiny neuron", NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE3)

In [ ]:
#Obs, Null, Z, P = test_supercluster_enrichment("CGE interneuron", SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
CT = "CGE interneuron"  
Obs, Null, Z, P = test_supercluster_enrichment(CT, NDD_Bias, NDD_Perm_Z2_BiasDFs_TYPE3)
#CT = "MGE interneuron"
#Obs, Null, Z, P = test_supercluster_enrichment(CT, SCZ_Bias, SCZ_Perm_Z2_BiasDFs_TYPE4)
plt.figure(figsize=(6,4))
plt.hist(Null, bins=30, alpha=0.5, label='Null')
plt.axvline(Obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Enrichment Score')
plt.ylabel('Count')
plt.title('{} [Type 3 Perm]\nP = {:.3f}'.format(CT, P))
plt.legend()
plt.tight_layout()

# End Test

In [ ]:
HCT_Z2_MAT_HCT_clip = pd.read_csv("../dat/HumanCTExpressionMats/Human.CT.Exp.Entrez.log2.Z2.HCT.z1clip3.csv", index_col=0)
max_Z, min_Z = 3, -3
HCT_Z2_MAT_HCT_clip = HCT_Z2_MAT_HCT_clip.clip(upper=max_Z, lower=min_Z)

In [ ]:
HCT_Z2_MAT_HCT = pd.read_csv("../dat/HumanCTExpressionMats/Human.CT.Exp.Entrez.log2.Z2.HCT.csv", index_col=0)
max_Z, min_Z = 5, -5
HCT_Z2_MAT_HCT = HCT_Z2_MAT_HCT.clip(upper=max_Z, lower=min_Z)

In [ ]:
ASD_GW = Fil2Dict("../../ASD_Circuits/dat/Unionize_bias/Spark_Meta_EWS.GeneWeight.csv")
SCZ_GW = Fil2Dict("../dat3/SCZ.top61.gw.csv")
X22q_GW = Fil2Dict("../dat3/22q.v2.gw.csv")

In [ ]:
BGMR = pd.read_csv("~/Work/Resources/MutationRate_20170710_rate.txt", delimiter="\t")
BGMR["Entrez"] = [int(GeneSymbol2Entrez.get(x, -1)) for x in BGMR["GeneName"].values]
BGMR = BGMR[BGMR["Entrez"].isin(HCT_Z2_MAT_HCT.index.values)]
BGMR.index = BGMR["Entrez"].values
#BGMR["Mutability_Dmg"] = BGMR["p_LGD"] + BGMR["prevel_0.5"]
#BGMR["Mutability_syn"] = BGMR["p_synonymous"]

In [ ]:
Spark_Denovo = pd.read_excel("../../ASD_Circuits/notebooks_v2/dat_in/41588_2022_1148_MOESM4_ESM.xlsx",
                           skiprows=2, sheet_name="Table S7")
Spark_Denovo = Spark_Denovo[Spark_Denovo[
    "pDenovoWEST_Meta"]!="."]
Spark_Denovo_ExomeWide = Spark_Denovo[Spark_Denovo[
    "pDenovoWEST_Meta"]<=2.5e-6]
Spark_Denovo_ExomeWide.shape

In [ ]:
Spark_Denovo_ExomeWide.to_csv("../dat/GeneWeights/ASD.ExomeWide.Denovo.csv", index=False)

In [ ]:
Mut_n_IQ = pd.read_csv("../dat/ASD_IQ_Mut.csv")
Mut_n_IQ = Mut_n_IQ[Mut_n_IQ["Entrez"].isin(HCT_Z2_MAT_HCT.index.values)]
Mut_n_IQ.columns.values

In [ ]:
top_Genes = Spark_Denovo.head(61)["HGNC"].values
Mut_n_IQ_conf = Mut_n_IQ[Mut_n_IQ["HGNC"].isin(top_Genes)]
Mut_n_IQ_conf.shape

In [ ]:
HighIQMuts = Mut_n_IQ_conf[Mut_n_IQ_conf["IQ"]>70]
LowIQMuts = Mut_n_IQ_conf[Mut_n_IQ_conf["IQ"]<=70]

In [ ]:
HighIQMuts.head(2)

In [ ]:
HighIQMuts.to_csv("../dat/GeneWeights/ASD.HighIQ.Mutable.csv", index=False)
LowIQMuts.to_csv("../dat/GeneWeights/ASD.LowIQ.Mutable.csv", index=False)
Mut_n_IQ_conf.to_csv("../dat/GeneWeights/ASD.All.Mutable.csv", index=False)

In [ ]:
HighIQMuts["Entrez"].unique()

In [ ]:
def normalize(probs):
    prob_factor = 1 / sum(probs)
    return [prob_factor * p for p in probs]
def Reduce(List, Counts):
    res = []
    Counts = list(Counts)
    for L in List:
        res.append(Counts.count(L))
    return res
def Table2GW_ASD(Mutable):
    gene2MutN = {}
    for g, row in Mutable.iterrows():
        gene2MutN[g] = row["NLGD"] * 0.554 + row["NDmis"] * 0.333
    return gene2MutN

def SingleSet_ReWeights_ASD(DZ_Muts, BGMR):
    N_LGD = DZ_Muts[DZ_Muts["GeneEff"]!="missense"].shape[0]
    N_Dmis = DZ_Muts[DZ_Muts["GeneEff"]=="missense"].shape[0]
    
    Genes = DZ_Muts["Entrez"].unique()
    LGD_P = []
    Dmis_P = []
    for g in Genes:
        try:
            LGD_P.append(BGMR.loc[g, "p_LGD"])
            Dmis_P.append(BGMR.loc[g, "prevel_0.5"])
        except:
            LGD_P.append(BGMR["p_LGD"].mean())
            Dmis_P.append(BGMR["prevel_0.5"].mean())
            
    Permed_LGD = np.random.choice(Genes, size=N_LGD, p=normalize(LGD_P))
    Permed_LGD_Count = Reduce(Genes, Permed_LGD)
    Permed_Dmis = np.random.choice(Genes, size=N_Dmis, p=normalize(Dmis_P))
    Permed_Dmis_Count = Reduce(Genes, Permed_Dmis)
    tmp_Mutable = pd.DataFrame(data={"Entrez":Genes, "NLGD":Permed_LGD_Count,
                                    "NDmis":Permed_Dmis_Count})
    tmp_Mutable = tmp_Mutable.set_index("Entrez")
    perm_GW = Table2GW_ASD(tmp_Mutable)
    return perm_GW

def Table2GW_SCZ(Mutable):
    gene2MutN = {}
    for g, row in Mutable.iterrows():
        gene2MutN[g] = row["NLGD"] * 0.554 + row["NDmis"] * 0.333
    return gene2MutN

def SingleSet_ReWeight_SCZ(SCZ_MutDF):
    #Mut_types = ["PTV", "mis3", "mis2"]
    N_PTV = SCZ_MutDF["Case PTV"].sum()
    N_mis3 = SCZ_MutDF["Case mis3"].sum()
    N_mis2 = SCZ_MutDF["Case mis2"].sum()
    
    Genes = SCZ_MutDF["Entrez"].unique()
    PTV_P, mis3_P, mis3_P = [],[],[],
    for g in Genes:
        PTV_P.append(SCZ_MutDF.loc[g, "Ctrl PTV"]+1)
        mis3_P.append(SCZ_MutDF.loc[g, "Ctrl mis3"]+1)
        mis2_P.append(SCZ_MutDF.loc[g, "Ctrl mis2"]+1)
            
    Permed_LGD = np.random.choice(Genes, size=N_PTV, p=normalize(PTV_P))
    Permed_LGD_Count = Reduce(Genes, Permed_LGD)
    Permed_mis3 = np.random.choice(Genes, size=N_mis3, p=normalize(mis3_P))
    Permed_mis3_Count = Reduce(Genes, Permed_Dmis)
    Permed_mis2 = np.random.choice(Genes, size=N_mis2, p=normalize(mis2_P))
    Permed_mis2_Count = Reduce(Genes, Permed_Dmis)
    
    tmp_Mutable = pd.DataFrame(data={"Entrez":Genes, "Case PTV":Permed_LGD_Count,
                                    "Case mis3":Permed_mis3_Count, "Case mis2":Permed_mis2_Count})
    tmp_Mutable = tmp_Mutable.set_index("Entrez")
    perm_GW = Table2GW_SCZ(tmp_Mutable)
    return perm_GW

In [ ]:
perm_GW = SingleSet_ReWeights_ASD(HighIQMuts, BGMR)
perm_GW_Bias = AvgCTZ_Weighted(HCT_Z2_MAT_HCT, perm_GW, Method = 1)
perm_GW_Bias = AnnotateCTDat(perm_GW_Bias, Anno)

In [ ]:
SuperClusterBias_BoxPlot(perm_GW_Bias, "perm_GW_Bias")

### For SCZ

In [ ]:
GeneDF = pd.read_excel("/home/jw3514/Work/ASD_Circuits/dat/genes/scz/41586_2022_4556_MOESM3_ESM.xlsx",
                    sheet_name="Table S5 - Gene Results")
ExAC_pLI = pd.read_csv("/home/jw3514/Work/Resources/gnomad.v2.1.1.lof_metrics.by_gene.txt", sep="\t",
                      index_col="gene")

In [ ]:
def oddsratio(NcaseMut, NctrlMut, dnvCount, Ncase = 24248, Nctrl=97322):
    if dnvCount!=dnvCount:
        dnvCount = 0
    NcaseMut += 1
    NctrlMut += 1
    AD = (NcaseMut) * (Nctrl-NctrlMut) 
    BC = (NctrlMut) * (Ncase-NcaseMut)
    return AD/BC + dnvCount

def Penetrance(NcaseMut, NctrlMut, Ncase = 24248, Nctrl=97322, prevelence=0.45/100):
    NcaseMut += 1
    NctrlMut += 1
    Ntotal_ =  Ncase/prevelence
    Nctrl_ = Ntotal_ - Ncase
    NctrlMut_ = Nctrl_*(NctrlMut/Nctrl)
    #print(Nctrl_, NctrlMut_)
    p = NcaseMut/(NcaseMut+NctrlMut_)
    return p

In [ ]:
def ModifyMutCount(CaseCount, ContCount, dnvCount, CaseN=24248, ContN=97322):
    if dnvCount!=dnvCount:
        dnvCount = 0
    return max(CaseCount - ContCount / ContN * CaseN + dnvCount, 0)

for i, row in GeneDF.iterrows():
    symbol = GeneDF.loc[i, "Gene Symbol"]
    try:
        GeneDF.loc[i, "Entrez"] = int(GeneSymbol2Entrez[symbol])
    except:
        GeneDF.loc[i, "Entrez"] = None
    try:
        GeneDF.loc[i, "pLI"] = ExAC_pLI.loc[symbol, "pLI"]
    except:
        GeneDF.loc[i, "pLI"] = 0
        
    
    GeneDF.loc[i, "nLGD"] = ModifyMutCount(GeneDF.loc[i, "Case PTV"], 
                                    GeneDF.loc[i, "Ctrl PTV"], GeneDF.loc[i, "De novo PTV"]) 
    GeneDF.loc[i, "nMis3"] = ModifyMutCount(GeneDF.loc[i, "Case mis3"], 
                                    GeneDF.loc[i, "Ctrl mis3"], GeneDF.loc[i, "De novo mis3"]) 
    GeneDF.loc[i, "nMis2"] = ModifyMutCount(GeneDF.loc[i, "Case mis2"], 
                                    GeneDF.loc[i, "Ctrl mis2"], GeneDF.loc[i, "De novo mis2"]) 
    GeneDF.loc[i, "LGD_OR"] = oddsratio(GeneDF.loc[i, "Case PTV"], 
                                    GeneDF.loc[i, "Ctrl PTV"], GeneDF.loc[i, "De novo PTV"])
    GeneDF.loc[i, "Mis3_OR"] = oddsratio(GeneDF.loc[i, "Case mis3"], 
                                    GeneDF.loc[i, "Ctrl mis3"], GeneDF.loc[i, "De novo mis3"])
    GeneDF.loc[i, "Mis2_OR"] = oddsratio(GeneDF.loc[i, "Case mis2"], 
                                    GeneDF.loc[i, "Ctrl mis2"], GeneDF.loc[i, "De novo mis2"])
    
    GeneDF.loc[i, "LGD_pen"] = Penetrance(GeneDF.loc[i, "Case PTV"], 
                                    GeneDF.loc[i, "Ctrl PTV"])
    GeneDF.loc[i, "Mis3_pen"] = Penetrance(GeneDF.loc[i, "Case mis3"], 
                                    GeneDF.loc[i, "Ctrl mis3"])
    GeneDF.loc[i, "Mis2_pen"] = Penetrance(GeneDF.loc[i, "Case mis2"], 
                                    GeneDF.loc[i, "Ctrl mis2"])

GeneDF = GeneDF.dropna(subset="Entrez")
GeneDF = GeneDF.set_index("Entrez")
GeneDF = GeneDF[GeneDF.index.isin(CT_Z2_MAT_HC.index.values)]
GeneDF.to_csv("../dat3/SCZ.ALLGENE.MutCountModified.csv")
GeneDF.shape

In [ ]:
def Gene_Weights_SCZ(MutFil, allen_mouse_genes, usepLI=False, Bmis=False, out=None):
    gene2MutN = {}
    for i, row in MutFil.iterrows():
        try:
            g = int(i)
            if g not in allen_mouse_genes:
                print(g, "not in allen mouse dataset")
                continue
        except:
            print(g, "Error converting Entrez ID")
        if usepLI:
            try:
                pLI = float(row["pLI"])
            except:
                print(g, "don't have pLI score on file, set to 0")
                pLI = 0.0
            if pLI >= 0.5:
                gene2MutN[g] = row["nLGD"] * 0.26 + row["nMis3"] * 0.25 + row["nMis2"] * 0.06  
            else:
                gene2MutN[g] = row["nLGD"] * 0.01 + row["nMis3"] * 0.01 + row["nMis2"] * 0 
        else:
            #gene2MutN[g] = row["nLGD"] * 0.26 + row["nMis3"] * 0.25 + row["nMis2"] * 0.06
            gene2MutN[g] = row["Case PTV"] * 0.26 + row["Case mis3"] * 0.25 + row["Case mis2"] * 0.06
    if out != None:
        writer = csv.writer(open(out, 'wt'))
        for k,v in sorted(gene2MutN.items(), key=lambda x:x[1], reverse=True):
           writer.writerow([k,v]) 
    return gene2MutN

In [ ]:
GeneDF.head(61).to_csv("../dat/GeneWeights/SCZ.top61.Mutable.csv")

In [ ]:
SCZ_GW = Gene_Weights_SCZ(GeneDF.head(61), HCT_Z2_MAT_HCT.index.values, usepLI=False)
SCZ_Bias = AvgCTZ_Weighted(HCT_Z2_MAT_HCT_clip, SCZ_GW, Method = 1)
SCZ_Bias = AnnotateCTDat(SCZ_Bias, Anno)

In [ ]:
SuperClusterBias_BoxPlot(SCZ_Bias, "SCZ Bias")

In [ ]:
def Table2GW_SCZ(Mutable):
    gene2MutN = {}
    for g, row in Mutable.iterrows():
        gene2MutN[g] = row["Case PTV"] * 0.26 + row["Case mis3"] * 0.25 + row["Case mis2"] * 0.06
    return gene2MutN
def SingleSet_ReWeight_SCZ(SCZ_MutDF):
    #Mut_types = ["PTV", "mis3", "mis2"]
    N_PTV = SCZ_MutDF["Case PTV"].sum().astype('int')
    N_mis3 = SCZ_MutDF["Case mis3"].sum().astype('int')
    N_mis2 = SCZ_MutDF["Case mis2"].sum().astype('int')
    
    Genes = SCZ_MutDF.index.unique()
    PTV_P, mis3_P, mis2_P = [],[],[],
    for g in Genes:
        PTV_P.append(SCZ_MutDF.loc[g, "Ctrl PTV"]+1)
        mis3_P.append(SCZ_MutDF.loc[g, "Ctrl mis3"]+1)
        mis2_P.append(SCZ_MutDF.loc[g, "Ctrl mis2"]+1)
            
    Permed_PTV = np.random.choice(Genes, size=N_PTV, p=normalize(PTV_P))
    Permed_PTV_Count = Reduce(Genes, Permed_PTV)
    Permed_mis3 = np.random.choice(Genes, size=N_mis3, p=normalize(mis3_P))
    Permed_mis3_Count = Reduce(Genes, Permed_mis3)
    Permed_mis2 = np.random.choice(Genes, size=N_mis2, p=normalize(mis2_P))
    Permed_mis2_Count = Reduce(Genes, Permed_mis2)
    
    tmp_Mutable = pd.DataFrame(data={"Entrez":Genes, "Case PTV":Permed_PTV_Count,
                                    "Case mis3":Permed_mis3_Count, "Case mis2":Permed_mis2_Count})
    tmp_Mutable = tmp_Mutable.set_index("Entrez")
    perm_GW = Table2GW_SCZ(tmp_Mutable)
    return perm_GW

In [ ]:
SCZ_RW_GW = SingleSet_ReWeight_SCZ(GeneDF.head(61))

In [ ]:
SCZ_RW_GW = SingleSet_ReWeight_SCZ(GeneDF.head(61))
RW_SCZ_Bias = AvgCTZ_Weighted(HCT_Z2_MAT_HCT_clip, SCZ_RW_GW)
RW_SCZ_Bias = AnnotateCTDat(RW_SCZ_Bias, Anno)

In [ ]:
SuperClusterBias_BoxPlot(RW_SCZ_Bias, "RW SCZ Bias")

In [ ]:
SCZ_GW = Gene_Weights_SCZ(GeneDF.head(61), HCT_Z2_MAT_HCT.index.values, usepLI=False)
SCZ_Bias = AvgCTZ_Weighted(HCT_Z2_MAT_HCT_clip, SCZ_GW, Method = 1, NonNeg=False)
SCZ_Bias = AnnotateCTDat(SCZ_Bias, Anno)

In [ ]:
SuperClusterBias_BoxPlot(SCZ_Bias, "SCZ Bias")

#### All CT GT Test

In [ ]:
def getBiasesBySTR(STR, dfs):
    biases = []
    for df in dfs:
        bias = df.loc[STR, "EFFECT"]
        biases.append(bias)
    biases = np.array(biases)
    return biases
def AddPvalue(DF, ContDF):
    for CT, row in DF.iterrows():
        mat_bias = getBiasesBySTR(CT, ContDF)
        Z, P = GetPermutationP(mat_bias, row["EFFECT"])
        DF.loc[CT, "EFFECT2"] = row["EFFECT"] - np.mean(mat_bias)
        DF.loc[CT, "Pvalue"] = P
        DF.loc[CT, "Z_Match"] = Z
        DF.loc[CT, "Z_Pvalue"] = scipy.stats.norm.sf(abs(Z))

    acc, qvalues = stats.multitest.fdrcorrection(DF["Pvalue"].values, alpha=0.1,
                                                       method="i")
    print(sum(acc))
    DF["qvalues"] = qvalues
    #ASD_Bias.to_csv("../dat/Unionize_bias/Spark_Meta_EWS.Z2.bias.subsib.FDR.csv")

    acc, qvalues = stats.multitest.fdrcorrection(DF["Z_Pvalue"].values, alpha=0.1,
                                                   method="i")
    print(sum(acc))
    DF["qvalues_ZP"] = qvalues
    return DF
def LoadCTRLBiasDFs(CTRL_DIR, n_samples=10000):
    CTRLBiasDFs = []
    for i in range(n_samples):
        try:
            DF = pd.read_csv("{}/cont.bias.{}.csv.gz".format(CTRL_DIR, i), index_col=0)
            CTRLBiasDFs.append(DF)
        except:
            continue
    print(len(CTRLBiasDFs))
    return CTRLBiasDFs
def PlotQQ(DFList, NameList):
    plt.figure(dpi=300, figsize=(10, 8))
    sns.set(style="whitegrid", context="talk")
    for DF, name in zip(DFList, NameList):
        DF_Pvals = DF["Pvalue"].values
        Qexp, Qobs = GetExpQ(DF_Pvals)
        plt.scatter(Qexp, Qobs , alpha=0.7, s=50, label=name)
    max_val = 4.5
    plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2)
    plt.xlabel('Expected -log10(p-value)', fontsize=25, weight='bold')
    plt.ylabel('Observed -log10(p-value)', fontsize=25, weight='bold')
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    # Add grid
    plt.grid(True, linestyle='--', alpha=0.5)
    # Tight layout
    plt.tight_layout()
    plt.legend(fontsize=15, loc="lower right")
    #plt.title()
    plt.show()

In [ ]:
RandGDFs_61G_GW = LoadCTRLBiasDFs("/home/jw3514/Work/CellType_Psy/dat/CTRL/RandGenes/RandGene_60_z1clip3.3")

In [ ]:
RandGDFs_61G_Unif = LoadCTRLBiasDFs("/home/jw3514/Work/CellType_Psy/dat/CTRL/RandGenes/UniformWeights/RandGene_60_z1clip3.3/")

##### CGE

In [ ]:
ASD_HIQ_GW_Unif = dict(zip(HIQ_GW.keys(), [1]*len(HIQ_GW.keys())))
SCZ_GW_Unif =  dict(zip(SCZ_GW.keys(), [1]*len(SCZ_GW.keys())))

In [ ]:
ASD_HIQ_Unif_Bias = AvgCTZ_Weighted(HCT_Z2_MAT_HCT_clip, ASD_HIQ_GW_Unif)
ASD_HIQ_Unif_Bias = AnnotateCTDat(ASD_HIQ_Unif_Bias, Anno)
SCZ_top61_Unif_Bias = AvgCTZ_Weighted(HCT_Z2_MAT_HCT_clip, SCZ_GW_Unif)
SCZ_top61_Unif_Bias = AnnotateCTDat(SCZ_top61_Unif_Bias, Anno)

In [ ]:
SuperClusterBias_BoxPlot(SCZ_top61_Unif_Bias, "SCZ_top61_Unif_Bias")

In [ ]:
#SuperClusterBias_BoxPlot(SCZ_top61_Unif_Bias, "SCZ Unif Bias")

In [ ]:
#CT = "CGE interneuron"
#CT = "MGE interneuron"
CT = "Medium spiny neuron"
CT_idx = Anno[Anno["Supercluster"]==CT].index.values
Required = len(CT_idx)
print(Required)

In [ ]:
ASD_CT_Bias_Array = HIQ_ASD_Bias.loc[CT_idx, "EFFECT"]
SCZ_CT_Bias_Array = SCZ_Bias.loc[CT_idx, "EFFECT"]
Obs_Test = ASD_CT_Bias_Array.sum() - SCZ_CT_Bias_Array.sum()
print(Obs_Test)

In [ ]:
ASD_CT_Bias_Array = ASD_HIQ_Unif_Bias.loc[CT_idx, "EFFECT"]
SCZ_CT_Bias_Array = SCZ_top61_Unif_Bias.loc[CT_idx, "EFFECT"]
Obs_Test = ASD_CT_Bias_Array.sum() - SCZ_CT_Bias_Array.sum()
print(Obs_Test)

In [ ]:
Diff_with_weight = []
Diff_without_weight = []
for i in range(1000):
    ## with Weights:
    DF1 = RandGDFs_61G_GW[i]
    DF2 = RandGDFs_61G_GW[i+4999]
    DF1_CT_Bias_array = DF1.loc[CT_idx, "EFFECT"]
    DF2_CT_Bias_array = DF2.loc[CT_idx, "EFFECT"]
    _Diff_with_weight = DF1_CT_Bias_array.sum() - DF2_CT_Bias_array.sum()
    Diff_with_weight.append(_Diff_with_weight)
    
    ## with out Weights:
    DF1 = RandGDFs_61G_Unif[i]
    DF2 = RandGDFs_61G_Unif[i+4999]
    DF1_CT_Bias_array = DF1.loc[CT_idx, "EFFECT"]
    DF2_CT_Bias_array = DF2.loc[CT_idx, "EFFECT"]
    _Diff_without_weight = DF1_CT_Bias_array.sum() - DF2_CT_Bias_array.sum()
    Diff_without_weight.append(_Diff_without_weight)

In [ ]:
sns.kdeplot(Diff_with_weight, label="with weight")
sns.kdeplot(Diff_without_weight, label="without weight")
plt.axvline(x=Obs_Test, color='red', linestyle='--', linewidth=2, label='obs')
plt.legend()

In [ ]:
#idx = 14
#idx = 15
idx = 16
DF1 = RandGDFs_61G_Unif[idx]
DF2 = RandGDFs_61G_Unif[idx+4999]
DF1_CT_Bias_array = DF1.loc[CT_idx, "EFFECT"]
DF2_CT_Bias_array = DF2.loc[CT_idx, "EFFECT"]
_Diff_with_weight = DF1_CT_Bias_array.sum() - DF2_CT_Bias_array.sum()
print(_Diff_with_weight)
plt.scatter(DF1_CT_Bias_array, DF2_CT_Bias_array)
#plt.

##### end

In [ ]:
Neurons

In [ ]:
#CT = "CGE interneuron"
#CT = "MGE interneuron"
CT = "Medium spiny neuron"
#CT = "Amygdala excitatory"
#CT = "Splatter"
#CT = "Cerebellar inhibitory"
#CT = "Mammillary body"
CT_idx = Anno[Anno["Supercluster"]==CT].index.values
Required = len(CT_idx)
print(Required)

In [ ]:
ASD_CT_Bias_Array = HIQ_ASD_Bias.loc[CT_idx, "EFFECT"]
SCZ_CT_Bias_Array = SCZ_Bias.loc[CT_idx, "EFFECT"]

In [ ]:
stats_scz = []
stats_asd = []
Test_DiffArray = SCZ_CT_Bias_Array - ASD_CT_Bias_Array
Total_tests = 10000
for i in range(Total_tests):
    DF = RandGDFs_61G_GW[i]
    DF_CT_Bias_array = DF.loc[CT_idx, "EFFECT"]
    stats_asd.append((DF_CT_Bias_array >= ASD_CT_Bias_Array).sum())
    stats_scz.append((DF_CT_Bias_array >= SCZ_CT_Bias_Array).sum())

In [ ]:
plt.hist(stats_asd, color="red", alpha=0.5, label="ASD")
plt.hist(stats_scz, color="blue", alpha=0.5, label="SCZ")
plt.show()
#plt.legend()
Pass_asd = (np.array(stats_asd) >= Required*0.9).sum()
Pass_scz = (np.array(stats_scz) >= Required*0.9).sum()
print(Pass_asd/Total_tests)
print(Pass_scz/Total_tests)

In [ ]:
Diffstats = []
Test_DiffArray = SCZ_CT_Bias_Array - ASD_CT_Bias_Array
#Test_DiffArray = ASD_CT_Bias_Array - SCZ_CT_Bias_Array
for i in range(4000):
    DF1 = RandGDFs_61G_GW[i]
    DF2 = RandGDFs_61G_GW[i+4999]
    DF1_CT_Bias_array = DF1.loc[CT_idx, "EFFECT"]
    DF2_CT_Bias_array = DF2.loc[CT_idx, "EFFECT"]
    Diff = DF1_CT_Bias_array - DF2_CT_Bias_array
    N_diff = (Diff >= Test_DiffArray).sum()
    Diffstats.append(N_diff)
plt.hist(Diffstats)
plt.show()
Pass = (np.array(Diffstats) >= Required-1).sum()
print(Pass/4000)

In [ ]:
Diffstats = []
#Test_DiffArray = SCZ_CT_Bias_Array - ASD_CT_Bias_Array
Test_DiffArray = ASD_CT_Bias_Array - SCZ_CT_Bias_Array
for i in range(4000):
    DF1 = RandGDFs_61G_GW[i]
    DF2 = RandGDFs_61G_GW[i+4999]
    DF1_CT_Bias_array = DF1.loc[CT_idx, "EFFECT"]
    DF2_CT_Bias_array = DF2.loc[CT_idx, "EFFECT"]
    Diff = DF1_CT_Bias_array - DF2_CT_Bias_array
    N_diff = (Diff >= Test_DiffArray).sum()
    Diffstats.append(N_diff)
plt.hist(Diffstats)
plt.show()
Pass = (np.array(Diffstats) >= Required-1).sum()
print(Pass/4000)

In [ ]:
Test_DiffArray

In [ ]:
HIQ_ASD_Bias = pd.read_csv("../dat/Bias/HCT.ASD.HIQ61.Z2.HCT.csv", index_col=0)
SCZ_Bias = pd.read_csv("../dat/Bias/HCT.SCZ61.Z2.HCT.csv", index_col=0)

In [ ]:
CT = "CGE interneuron"
CT = "MGE interneuron"
CT = "Medium Spiney Neurons"
CT_idx = Anno[Anno["Supercluster"]==CT].index.values
Required = len(CT_idx)

In [ ]:
ASD_CT_Bias_Array = HIQ_ASD_Bias.loc[CT_idx, "EFFECT"]
SCZ_CT_Bias_Array = SCZ_Bias.loc[CT_idx, "EFFECT"]
Test_DiffArray = SCZ_CT_Bias_Array - ASD_CT_Bias_Array

In [ ]:
stats = []
for i in range(4000):
    DF1 = RandGDFs_61G_GW[i]
    DF2 = RandGDFs_61G_GW[i+4999]
    DF1_CT_Bias_array = DF1.loc[CT_idx, "EFFECT"]
    DF2_CT_Bias_array = DF2.loc[CT_idx, "EFFECT"]
    Diff = DF1_CT_Bias_array - DF2_CT_Bias_array
    N_diff = (Diff >= Test_DiffArray).sum()
    stats.append(N_diff)

In [ ]:
plt.hist(stats)
plt.show()
Pass = (np.array(stats) >= Required).sum()
print(Pass/4000)

In [ ]:
#ASD_GW = Fil2Dict("../dat/GeneWeights/")
HIQ_GW = Fil2Dict("../../ASD_Circuits/dat/Unionize_bias/ASD.HIQ.gw.csv")
LIQ_GW = Fil2Dict("../../ASD_Circuits/dat/Unionize_bias/ASD.LIQ.gw.csv")
SCZ_GW = Fil2Dict("../dat3/SCZ_MutCount_61.gw")

In [ ]:
def GeneRand(GW1, GW2):
    GW1_keys = list(GW1.keys())
    GW1_Weights = list(GW1.values())
    
    GW2_keys = list(GW2.keys())
    GW2_Weights = list(GW2.values())
    
    AB = list(GW1_keys) + list(GW2_keys)
    
    random.shuffle(AB)
    Ap = AB[:len(GW1_keys)]
    Bp = AB[len(GW2_keys):]
    
    GW1_Rand = dict(zip(Ap, GW1_Weights))
    GW2_Rand = dict(zip(Bp, GW2_Weights))
    
    return GW1_Rand, GW2_Rand

In [ ]:
GenePermDF1, GenePermDF2 = [], []
for i in range(1000):
    GW1, GW2 = GeneRand(HIQ_GW, SCZ_GW)
    DF1 = AvgCTZ_Weighted(HCT_Z2_MAT_HCT_clip, GW1, Method = 1)
    DF2 = AvgCTZ_Weighted(HCT_Z2_MAT_HCT_clip, GW2, Method = 1)
    GenePermDF1.append(DF1)
    GenePermDF2.append(DF2)

In [ ]:
print(len(GenePermDF1), len(GenePermDF2))

In [ ]:
N_test = len(GenePermDF1)

In [ ]:
#CT = "CGE interneuron"
#CT = "MGE interneuron"
CT = "Medium spiny neuron"
CT_idx = Anno[Anno["Supercluster"]==CT].index.values
Required = len(CT_idx)
ASD_CT_Bias_Array = HIQ_ASD_Bias.loc[CT_idx, "EFFECT"]
SCZ_CT_Bias_Array = SCZ_Bias.loc[CT_idx, "EFFECT"]
Test_DiffArray = (SCZ_CT_Bias_Array - ASD_CT_Bias_Array)
stats2_obs = Test_DiffArray.sum()
print(Required, stats2_obs)

In [ ]:
stats = []
stats2 = []
for i in range(N_test):
    DF1 = GenePermDF1[i]
    DF2 = GenePermDF2[i]
    DF1.index = [int(x) for x in DF1.index.values]
    DF2.index = [int(x) for x in DF2.index.values]
    DF1_CT_Bias_array = DF1.loc[CT_idx, "EFFECT"]
    DF2_CT_Bias_array = DF2.loc[CT_idx, "EFFECT"]
    Diff = DF1_CT_Bias_array - DF2_CT_Bias_array
    N_diff = (Diff >= Test_DiffArray).sum()
    stats2.append(Diff.sum())
    stats.append(N_diff)

In [ ]:
plt.hist(stats)
plt.show()
Pass = (np.array(stats) >= Required).sum()
print(Pass/N_test)

In [ ]:
plt.hist(stats2)
plt.show()
Pass = (np.array(stats2) >= stats2_obs).sum()
print(Pass/N_test)

#### Mutation Redistribution

In [ ]:
Permut_ASD_DF_MutPerm_CT = []
Permut_SCZ_DF_MutPerm_CT = []
#DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/BiasDiff_RW/HIQASD_SCZ/"
#DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/BiasDiff_RW/HIQASD_SCZ_v2/"
#DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/BiasDiff_RW/HIQASD_SCZ_v3/"
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/BiasDiff_RW/HIQASD_SCZ_v4/"
N=10000
for i in range(N):
    try:
        DF1 = pd.read_csv("{}/{}.ASD.perm.bias.csv".format(DIR, i), index_col=0)
        DF2 = pd.read_csv("{}/{}.SCZ.perm.bias.csv".format(DIR, i), index_col=0)
        Permut_ASD_DF_MutPerm_CT.append(DF1)
        Permut_SCZ_DF_MutPerm_CT.append(DF2)
    except:
        continue
N_Perm = len(Permut_ASD_DF_MutPerm_CT)
print(N_Perm)

In [ ]:
ASD_CT_Bias = pd.read_csv("../dat/Bias/HCT.ASD61.HIQ.Z2.HCT.csv", index_col=0)
SCZ_CT_Bias = pd.read_csv("../dat/Bias/HCT.SCZ61.Z2.HCT.csv", index_col=0)

In [ ]:
def TestDiff_Perm(CT, ASD_CT_Bias, SCZ_CT_Bias, Permut_ASD_DF_MutPerm_CT, Permut_SCZ_DF_MutPerm_CT):
    CT_ASD_Bias = ASD_CT_Bias[ASD_CT_Bias["Supercluster"]==CT]
    CT_SCZ_Bias = SCZ_CT_Bias[SCZ_CT_Bias["Supercluster"]==CT]
    ct_ASD_Bias = CT_ASD_Bias["EFFECT"].sum()
    ct_SCZ_Bias = CT_SCZ_Bias["EFFECT"].sum()
    BiasDiff = ct_ASD_Bias - ct_SCZ_Bias
    #print(ct_ASD_Bias, ct_SCZ_Bias, BiasDiff)
    PermutedASD = []
    PermutedSCZ = []
    PermutedDiff = []
    for i in range(N_Perm):
        DF1 = Permut_ASD_DF_MutPerm_CT[i]
        DF2 = Permut_SCZ_DF_MutPerm_CT[i]
        DF1_Bias = DF1[DF1["Supercluster"]==CT]
        DF2_Bias = DF2[DF2["Supercluster"]==CT]
        _ct_ASD_Bias = DF1_Bias["EFFECT"].sum()
        _ct_SCZ_Bias = DF2_Bias["EFFECT"].sum()
        _BiasDiff = _ct_ASD_Bias - _ct_SCZ_Bias
        PermutedASD.append(_ct_ASD_Bias)
        PermutedSCZ.append(_ct_SCZ_Bias)
        PermutedDiff.append(_BiasDiff)
        
    P,Z = xxGetPermutationP(Obs = BiasDiff, Null=PermutedDiff)
    fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(24,6), dpi=120)

    PlotPermutationP(Obs = ct_ASD_Bias, ax=ax1, Null=PermutedASD)
    ax1.set_xlabel("ASD")

    PlotPermutationP(Obs = ct_SCZ_Bias, ax=ax2, Null=PermutedSCZ)
    ax2.set_xlabel("SCZ")

    PlotPermutationP(Obs = BiasDiff, ax=ax3, Null=PermutedDiff)
    ax3.set_xlabel("Bias Diff")
    #ax3.set_title(CT + " [Mut Perm]", fontsize=15)
    plt.tight_layout()
    plt.suptitle(CT, fontsize=30, y=1.02)
    plt.show()

In [ ]:
#CT = "CGE interneuron"
for CT in Neurons:
    TestDiff_Perm(CT, ASD_CT_Bias, SCZ_CT_Bias, Permut_ASD_DF_MutPerm_CT, Permut_SCZ_DF_MutPerm_CT)

In [ ]:
Permut_ASD_DF_MutPerm_v5 = []
Permut_SCZ_DF_MutPerm_v5 = []
DIR = "/home/jw3514/Work/CellType_Psy/dat/CTRL/BiasDiff_RW/HIQASD_SCZ_v5/"
N=10000
for i in range(N):
    try:
        DF1 = pd.read_csv("{}/{}.ASD.perm.bias.csv".format(DIR, i), index_col=0)
        DF2 = pd.read_csv("{}/{}.SCZ.perm.bias.csv".format(DIR, i), index_col=0)
        Permut_ASD_DF_MutPerm_v5.append(DF1)
        Permut_SCZ_DF_MutPerm_v5.append(DF2)
    except:
        continue
N_Perm = len(Permut_ASD_DF_MutPerm_v5)
print(N_Perm)

In [ ]:
CT = "CGE interneuron"
TestDiff_Perm(CT, ASD_CT_Bias, SCZ_CT_Bias, Permut_ASD_DF_MutPerm_v5, Permut_SCZ_DF_MutPerm_v5)

In [ ]:
CT = "MGE interneuron"
TestDiff_Perm(CT, ASD_CT_Bias, SCZ_CT_Bias, Permut_ASD_DF_MutPerm_v5, Permut_SCZ_DF_MutPerm_v5)

In [ ]:
CT = "Medium spiny neuron"
TestDiff_Perm(CT, ASD_CT_Bias, SCZ_CT_Bias, Permut_ASD_DF_MutPerm_v5, Permut_SCZ_DF_MutPerm_v5)

In [ ]:
#CT = "CGE interneuron"
for CT in Neurons:
    TestDiff_Perm(CT, ASD_CT_Bias, SCZ_CT_Bias, Permut_ASD_DF_MutPerm_v5, Permut_SCZ_DF_MutPerm_v5)